# Notebook 08: Interactive Dashboard Demo
## Gradio Interface: Upload CSV → Anomaly Score + SHAP Explanation

In [ ]:
import numpy as np, pandas as pd, struct
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = Path('/run/media/sham/AI_/ai-stack/projects/biopharma-contamination-detection')
OUT = BASE / 'data' / 'processed'
FIG = BASE / 'figures'; FIG.mkdir(exist_ok=True)
df = pd.read_parquet(OUT / 'real_dataset.parquet')

In [ ]:
def unpack(row):    n = row['n_wl']; data = struct.unpack(f'{n*2}d', row['spectrum_bytes'])    return np.array(data[::2]), np.array(data[1::2])target_wl = np.arange(230, 610, 1)def interp(row): return np.interp(target_wl, *unpack(row))X = np.vstack(df.apply(interp, axis=1).values)y = df['label'].valuesscaler = RobustScaler().fit(X[y==0])X_s = scaler.transform(X)iso = IsolationForest(n_estimators=200, random_state=42)iso.fit(X_s[y==0])print('Models loaded')

## Gradio Interface

In [ ]:
import gradio as gr

def predict(file):
    df_csv = pd.read_csv(file.name, skiprows=1)
    if 'Wavelength' in df_csv.columns and 'CorrectedAbs' in df_csv.columns:
        wl, ab = df_csv['Wavelength'].values, df_csv['CorrectedAbs'].values
    elif 'Wavelength' in df_csv.columns and 'Absorbance' in df_csv.columns:
        wl, ab = df_csv['Wavelength'].values, df_csv['Absorbance'].values
    else:
        wl, ab = df_csv.iloc[:,0].values, df_csv.iloc[:,1].values
    
    x = np.interp(target_wl, wl, ab).reshape(1, -1)
    x_s = scaler.transform(x)
    score = -iso.score_samples(x_s)[0]
    thresh = np.percentile(-iso.score_samples(X_s[y==0]), 95)
    is_anomaly = score > thresh
    confidence = min(score / thresh, 1.0)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(wl, ab, linewidth=1.5)
    ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('Absorbance')
    ax.set_title(f'Anomaly Score: {score:.4f} (threshold: {thresh:.4f})')
    path = str(FIG / 'prediction_plot.png')
    fig.savefig(path, dpi=150); plt.close()
    
    return (f"{'⚠️ CONTAMINATED' if is_anomaly else '✅ STERILE'} | Score: {score:.4f} | Confidence: {confidence:.1%}", path)

demo = gr.Interface(
    fn=predict,
    inputs=gr.File(label='Upload UV-Vis CSV'),
    outputs=[gr.Textbox(label='Result'), gr.Image(label='Spectrum')],
    title='Biopharma Contamination Detector',
    description='Upload a UV-Vis spectrum CSV to detect microbial contamination'
)
print('Dashboard defined. Launch with: demo.launch()')

## Save as standalone script

In [ ]:
script = '''
import gradio as gr
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib, os

BASE = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
print("Dashboard requires trained model. Run Notebook 03 first to save model.")
print("Then launch with: gradio app.py")
'''
with open(BASE + '/dashboard.py', 'w') as f: f.write(script)
print('Saved: dashboard.py')